# 03 — Feature Engineering

We build a complete `(product_id, date)` panel (filling any missing product-days with 0 demand so lag/rolling windows are computed on a continuous calendar), then add lag, rolling-window, and calendar features. See `src/features.py` for the reusable implementation used here and in later notebooks.

## Leakage rule

> Every feature for date **T** must be computable using only information that
> existed at or before date **T**.

Concretely:
- `lag_7` on Jan 20 = demand on Jan 13. Fine — that's the past.
- A feature built from Jan 21–25 demand to predict Jan 20 would use information
  that doesn't exist yet at prediction time — **that's leakage**, and it would make
  offline metrics look great while being impossible to reproduce in production.
- Rolling means are computed on `quantity.shift(1)` **before** taking the window,
  so `rolling_mean_7` on day T never includes day T's own demand.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from features import build_panel, add_features, FEATURE_COLUMNS, TARGET_COLUMN, LAGS, ROLLING_WINDOWS

df = pd.read_csv("../data/sales_clean.csv", parse_dates=["date"])
panel = build_panel(df)
print("Panel shape (product x date):", panel.shape)
panel.head()

Panel shape (product x date): (36480, 8)


        date product_id     category  ...  promotion  discount  holiday
0 2023-01-01       P001  Electronics  ...        0.0       0.0      1.0
1 2023-01-02       P001  Electronics  ...        0.0       0.0      0.0
2 2023-01-03       P001  Electronics  ...        0.0       0.0      0.0
3 2023-01-04       P001  Electronics  ...        0.0       0.0      0.0
4 2023-01-05       P001  Electronics  ...        0.0       0.0      0.0

[5 rows x 8 columns]

In [2]:
feat = add_features(panel)
print("Feature table shape:", feat.shape)
print("\nFeature columns:", FEATURE_COLUMNS)
feat[["product_id", "date", "quantity"] + FEATURE_COLUMNS].head(10)

Feature table shape: (36480, 22)

Feature columns: ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_14', 'rolling_std_28', 'day_of_week', 'month', 'week_of_year', 'is_weekend', 'price', 'promotion', 'discount', 'holiday']


  product_id       date  quantity  lag_1  ...  price  promotion  discount  holiday
0       P001 2023-01-01     176.0    NaN  ...  30.79        0.0       0.0      1.0
1       P001 2023-01-02      91.0  176.0  ...  30.79        0.0       0.0      0.0
2       P001 2023-01-03      86.0   91.0  ...  30.79        0.0       0.0      0.0
3       P001 2023-01-04      54.0   86.0  ...  30.79        0.0       0.0      0.0
4       P001 2023-01-05      51.0   54.0  ...  30.79        0.0       0.0      0.0
5       P001 2023-01-06     105.0   51.0  ...  30.79        0.0       0.0      0.0
6       P001 2023-01-07      29.0  105.0  ...  30.79        0.0       0.0      0.0
7       P001 2023-01-08     136.0   29.0  ...  30.79        0.0       0.0      0.0
8       P001 2023-01-09      79.0  136.0  ...  30.79        0.0       0.0      0.0
9       P001 2023-01-10     129.0   79.0  ...  30.79        0.0       0.0      0.0

[10 rows x 21 columns]

## Confirming the leakage rule holds

Spot-check: for a given product, `lag_7` on date T should equal the actual `quantity` on date T-7.

In [3]:
check = feat[feat["product_id"] == "P001"].set_index("date")["quantity"]
sample_date = check.index[40]
lag7_value = feat[(feat["product_id"] == "P001") & (feat["date"] == sample_date)]["lag_7"].iloc[0]
actual_7_days_before = check.loc[sample_date - pd.Timedelta(days=7)]
print(f"lag_7 on {sample_date.date()}:            {lag7_value}")
print(f"actual quantity 7 days earlier: {actual_7_days_before}")
assert lag7_value == actual_7_days_before
print("OK: lag_7 matches actual demand exactly 7 days earlier, nothing later leaks in.")

lag_7 on 2023-02-10:            119.0
actual quantity 7 days earlier: 119.0
OK: lag_7 matches actual demand exactly 7 days earlier, nothing later leaks in.


## Missing values from feature construction

The first `max(lags)` days of each product's history can't have all lag features (there's no 28-days-ago yet for day 5 of the dataset). We drop those rows before modeling — this is standard for lag-based forecasting and costs us a small, fixed number of early rows per product, not real data.

In [4]:
na_counts = feat[FEATURE_COLUMNS].isna().sum()
print(na_counts[na_counts > 0])

model_df = feat.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
print(f"\nRows before dropping warm-up NaNs: {len(feat)}")
print(f"Rows after:                        {len(model_df)}")
print(f"Rows lost per product (~expected): {(len(feat)-len(model_df)) / feat['product_id'].nunique():.1f}")

lag_1                40
lag_7               280
lag_14              560
lag_28             1120
rolling_mean_7      280
rolling_mean_14     560
rolling_mean_28    1120
rolling_std_7       280
rolling_std_14      560
rolling_std_28     1120
dtype: int64

Rows before dropping warm-up NaNs: 36480
Rows after:                        35360
Rows lost per product (~expected): 28.0


In [5]:
model_df.to_csv("../data/model_features.csv", index=False)
model_df[["product_id", "date", TARGET_COLUMN] + FEATURE_COLUMNS].describe().T

                   count                 mean  ...                  max        std
date               35360  2024-04-14 12:00:00  ...  2025-06-30 00:00:00        NaN
quantity         35360.0            72.144118  ...                443.0  42.889155
lag_1            35360.0            72.167053  ...                443.0  42.901686
lag_7            35360.0            72.121776  ...                443.0  42.884908
lag_14           35360.0            72.075141  ...                443.0  42.864314
lag_28           35360.0            71.967902  ...                509.0  42.977947
rolling_mean_7   35360.0            72.125105  ...           201.714286  35.725197
rolling_mean_14  35360.0            72.111654  ...           187.642857  35.186776
rolling_mean_28  35360.0            72.033973  ...           183.142857  34.835047
rolling_std_7    35360.0            21.264365  ...           132.811718  14.297956
rolling_std_14   35360.0            21.951979  ...            98.095491  12.839676
roll